# 08 — Избыточный SELECT *

> **`vuln_class`:** `SELECT_STAR` · **Риск:** 5/10 · **CWE-1295**

`SELECT *` — не уязвимость **сама по себе**, но классический «множитель» для других. Раскрывает в выборке **все** колонки таблицы, включая чувствительные, которые приложению/UI не нужны.


## 🧒 Аналогия для ребёнка

Тебе нужны **только конфеты** из коробки шоколада. Ты приходишь
и говоришь продавцу: «дай **всё**». Он даёт тебе всю коробку,
включая обёртки, разделители, описание состава, штрих-код.

- **Плохо:** ты тащишь домой всю коробку и аккуратно достаёшь
  конфеты. Лишнее — мусор, лишний вес, лишнее время.
- **Хорошо:** говоришь «дай две вишнёвые и одну с орехом» —
  получаешь ровно то, что нужно.

В БД: `SELECT *` тащит все колонки, **включая** `password_hash`,
`internal_notes`, `migration_token`. Тратится трафик, кеш, безопасность.


## 1. Setup


In [ ]:
"""
@brief Подготовка окружения и mock-БД через in-memory SQLite.
@details
    Никаких внешних зависимостей кроме stdlib + sqlite3 (есть в Colab из коробки).
    SQLite используем как «упрощённую модель PostgreSQL» — он умеет
    почти весь стандартный SQL, что достаточно для демонстраций уязвимостей.
@note
    Реальная система работает на PostgreSQL (см. ADR-0001),
    использует pglast для AST-парсинга. Здесь, для наглядности,
    эмулируем аудитор через `re` (регулярки) и простой pattern matching.
"""
import sqlite3
import re
import time
from textwrap import dedent


def section(title):
    """@brief Печатает заголовок секции."""
    print("\n" + "=" * 72)
    print(title)
    print("=" * 72)


def show_result(rows, max_rows=10):
    """@brief Печатает результаты запроса в виде таблицы."""
    if not rows:
        print("  (нет строк)")
        return
    for i, r in enumerate(rows[:max_rows]):
        print(f"  {i + 1:>3}. {r}")
    if len(rows) > max_rows:
        print(f"  ... ещё {len(rows) - max_rows} строк")


def print_finding(f):
    """@brief Красиво печатает Finding от нашего аудитора."""
    print(f"  ⚠️  {f['rule_id']}")
    print(f"      vuln_class:  {f['vuln_class']}")
    print(f"      severity:    {f['severity']}")
    print(f"      risk_score:  {f['risk_score']}/10")
    print(f"      message:     {f['message']}")
    if f.get("evidence_refs"):
        print(f"      ссылки:      {', '.join(f['evidence_refs'])}")


def setup_users_full():
    conn = sqlite3.connect(":memory:")
    conn.execute("""
        CREATE TABLE users (
            id              INTEGER PRIMARY KEY,
            login           TEXT,
            full_name       TEXT,
            email           TEXT,
            password_hash   TEXT,          -- ⚠️
            internal_notes  TEXT,          -- ⚠️ внутренние заметки (NDA)
            migration_token TEXT           -- ⚠️ кред
        )""")
    conn.executemany(
        "INSERT INTO users (login, full_name, email, password_hash, internal_notes, migration_token) "
        "VALUES (?, ?, ?, ?, ?, ?)",
        [
            ("admin",  "Анна А.", "anna@ex.com",  "h_admin",  "VIP клиент",   "tk_admin_xyz"),
            ("bob",    "Боб Б.",  "bob@ex.com",   "h_bob",    "Жалоба #123",  "tk_bob_qqq"),
        ],
    )
    conn.commit()
    return conn


conn = setup_users_full()


## 2. Уязвимый запрос — нужен список пользователей для UI


In [ ]:
##
# @brief УЯЗВИМАЯ функция: SELECT * — выбирает всё, что есть.
# @warning  Утечка password_hash, migration_token, internal_notes в UI/логи.
def list_users_BAD(conn):
    sql = "SELECT * FROM users"
    print(f"  SQL: {sql}")
    return conn.execute(sql).fetchall()


section("Что попадает в UI/логи")
for r in list_users_BAD(conn):
    print(f"  {r}")


## 3. Аудитор Phase 1 — `R001-select-star`


In [ ]:
##
# @brief Phase 1 R001 — детект SELECT *.
# @note  Игнорируем COUNT(*), row_to_json(t.*) обрабатывается отдельно (повышаем severity).
def audit_R001_select_star(sql_text):
    findings = []
    # SELECT * (с возможным алиасом t.*) — но НЕ COUNT(*) и НЕ внутри агрегата
    pattern = r"SELECT\s+(?:DISTINCT\s+)?(?:\w+\.)?\*"
    for m in re.finditer(pattern, sql_text, re.IGNORECASE):
        snippet = m.group(0)
        # Грубая проверка на COUNT/row_to_json
        ctx_start = max(0, m.start() - 25)
        ctx = sql_text[ctx_start:m.start()].lower()
        if "count(" in ctx or "row_to_json(" in ctx or "to_jsonb(" in ctx:
            continue
        findings.append({
            "rule_id":       "R001-select-star",
            "vuln_class":    "SELECT_STAR",
            "severity":      "medium", "risk_score": 5,
            "message":       f"{snippet!r} — выбираются все колонки, включая возможно чувствительные",
            "evidence_refs": ["CWE-1295"],
        })
    return findings


section("Аудитор по разным запросам")
for sql in [
    "SELECT * FROM users",
    "SELECT u.* FROM users u JOIN roles r ON u.role_id = r.id",
    "SELECT COUNT(*) FROM users",                         # это норм
    "SELECT id, login, full_name FROM users",             # это норм
]:
    print(f"\n SQL: {sql}")
    fs = audit_R001_select_star(sql)
    if fs:
        for f in fs:
            print_finding(f)
    else:
        print("  ✅ ok")


## 4. Phase 1b — раскрытие `*` через `information_schema.columns`

В проде sandbox-БД содержит схему. Раскрываем `*` в реальные колонки
и прогоняем правило `R009` (sensitive columns) — это даёт ещё один,
**более тяжёлый** finding.


In [ ]:
##
# @brief Имитация Phase 1b — раскрываем SELECT * через PRAGMA table_info
# @note  В PG было бы information_schema.columns, тут — PRAGMA SQLite.
def expand_star_and_check(conn, sql_text):
    m = re.match(r"\s*SELECT\s+\*\s+FROM\s+(\w+)", sql_text, re.IGNORECASE)
    if not m:
        return []
    table = m.group(1)
    cols = [r[1] for r in conn.execute(f"PRAGMA table_info({table})").fetchall()]
    print(f"  Раскрываем SELECT * → колонки: {cols}")
    # Прогоняем тот же regex R009
    suspicious = []
    for col in cols:
        for pat, sev, score in [
            (r"(?i)^(password|passwd|pwd|secret|api[_-]?key|token|access[_-]?token|migration[_-]?token)$", "critical", 8),
            (r"(?i)^internal(_notes|_data)?$", "medium", 6),
        ]:
            if re.match(pat, col):
                suspicious.append((col, sev, score))
                break
    return suspicious


susp = expand_star_and_check(conn, "SELECT * FROM users")
section("Phase 1b — чувствительные колонки в раскрытом *")
for col, sev, score in susp:
    print_finding({
        "rule_id":       "R001+R009-star-leaks-sensitive",
        "vuln_class":    "DIRECT_SENSITIVE",
        "severity":      sev, "risk_score": score,
        "message":       f"SELECT * раскроет {col!r} — это чувствительная колонка",
        "evidence_refs": ["CWE-200", "CWE-1295"],
    })


## 5. Безопасная версия — явный список колонок


In [ ]:
##
# @brief Безопасная функция: явный список колонок.
def list_users_GOOD(conn):
    sql = "SELECT id, login, full_name, email FROM users"
    return conn.execute(sql).fetchall()


section("Безопасная версия")
for r in list_users_GOOD(conn):
    print(f"  {r}")


## Итог

Мы увидели одно и то же на двух функциях:

- **Уязвимая** — украли данные / повредили БД / поднялись в правах.
- **Безопасная** — та же атака уходит в пустоту.

Между ними — **один аудитор** с конкретным правилом, которое можно
запустить детерминированно (без LLM) на каждом сгенерированном SQL.

## Куда дальше

- **Описание уязвимости (под микроскопом):** [problems/vulnerabilities/08-select-star/README.md](../problems/vulnerabilities/08-select-star/README.md)
- **Варианты решения + почему так:** [problems/vulnerabilities/08-select-star/solutions.md](../problems/vulnerabilities/08-select-star/solutions.md)
- **Архитектура цикла:** [docs/adr/0002-loop-architecture-langgraph.md](../docs/adr/0002-loop-architecture-langgraph.md)
- **Гибридный аудитор (pglast + LLM):** [docs/adr/0004-hybrid-auditor-ast-plus-llm.md](../docs/adr/0004-hybrid-auditor-ast-plus-llm.md)
